# Questão 4 - Análise de Clientes Fiéis

**Premissas obrigatórias:**
- Faturamento Total = soma de `total` por cliente
- Frequência = contagem de transações por cliente
- Ticket Médio = Faturamento Total / Frequência
- Diversidade de Categorias = qtd. de `category_id` distintos comprados
- Filtro de elite: >= 13 categorias distintas
- Desempate: `customer_id` crescente

In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
import pandas as pd
from src.db import get_engine

engine = get_engine()

## Questão 4.1 - SQL: Ticket Médio, Diversidade e Top 10 clientes fiéis

(ver `sql/queries/q4_clientes_fieis.sql`)

In [3]:
top10_query = """
WITH customer_metrics AS (
    SELECT
        customer_id,
        SUM(total)                              AS faturamento_total,
        COUNT(id)                               AS frequencia,
        ROUND(SUM(total) / COUNT(id), 2)        AS ticket_medio
    FROM orders
    GROUP BY customer_id
),
customer_categories AS (
    SELECT
        o.customer_id,
        COUNT(DISTINCT p.category_id) AS diversidade_categorias
    FROM orders o
    JOIN order_items oi      ON oi.order_id = o.id
    JOIN product_variants pv ON pv.id = oi.product_variant_id
    JOIN products p          ON p.id = pv.product_id
    GROUP BY o.customer_id
)
SELECT
    cm.customer_id,
    cm.faturamento_total,
    cm.frequencia,
    cm.ticket_medio,
    cc.diversidade_categorias
FROM customer_metrics cm
JOIN customer_categories cc ON cc.customer_id = cm.customer_id
WHERE cc.diversidade_categorias >= 13
ORDER BY cm.ticket_medio DESC, cm.customer_id ASC
LIMIT 10;
"""

top10_sql = pd.read_sql(top10_query, engine)
top10_sql

,customer_id,faturamento_total,frequencia,ticket_medio,diversidade_categorias
0,22,1087838.44,26,41839.94,14
1,1477,916262.58,22,41648.30,14
2,929,1082775.89,26,41645.23,14
3,1116,655737.20,16,40983.58,14
4,1691,815471.30,20,40773.57,14
5,774,726127.99,18,40340.44,14
6,1470,1040553.09,26,40021.27,14
7,1599,997616.46,25,39904.66,14
8,965,677297.78,17,39841.05,14
9,1722,1146455.22,29,39532.94,14


## Categoria mais vendida entre os 10 clientes fiéis

(maior `sum(quantity)` de itens comprados por categoria)

In [4]:
categoria_query = """
WITH customer_metrics AS (
    SELECT
        customer_id,
        SUM(total)                        AS faturamento_total,
        COUNT(id)                         AS frequencia,
        ROUND(SUM(total) / COUNT(id), 2)  AS ticket_medio
    FROM orders
    GROUP BY customer_id
),
customer_categories AS (
    SELECT
        o.customer_id,
        COUNT(DISTINCT p.category_id) AS diversidade_categorias
    FROM orders o
    JOIN order_items oi      ON oi.order_id = o.id
    JOIN product_variants pv ON pv.id = oi.product_variant_id
    JOIN products p          ON p.id = pv.product_id
    GROUP BY o.customer_id
),
top10_clientes_fieis AS (
    SELECT cm.customer_id, cm.ticket_medio, cc.diversidade_categorias
    FROM customer_metrics cm
    JOIN customer_categories cc ON cc.customer_id = cm.customer_id
    WHERE cc.diversidade_categorias >= 13
    ORDER BY cm.ticket_medio DESC, cm.customer_id ASC
    LIMIT 10
)
SELECT
    cat.name              AS categoria,
    SUM(oi.quantity)       AS total_itens_comprados
FROM top10_clientes_fieis t
JOIN orders o             ON o.customer_id = t.customer_id
JOIN order_items oi       ON oi.order_id = o.id
JOIN product_variants pv  ON pv.id = oi.product_variant_id
JOIN products p           ON p.id = pv.product_id
JOIN categories cat       ON cat.id = p.category_id
GROUP BY cat.name
ORDER BY total_itens_comprados DESC
LIMIT 1;
"""

categoria_vencedora_sql = pd.read_sql(categoria_query, engine)
categoria_vencedora_sql

,categoria,total_itens_comprados
0,Hélices,492


## Validação cruzada (pandas)

Reproduzindo a mesma lógica em pandas, a partir das tabelas já carregadas no banco, para confirmar os resultados do SQL por um caminho independente.

In [5]:
# Carregar tabelas do banco
orders = pd.read_sql("SELECT * FROM orders", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
product_variants = pd.read_sql("SELECT * FROM product_variants", engine)
products = pd.read_sql("SELECT * FROM products", engine)
categories = pd.read_sql("SELECT * FROM categories", engine)

# Faturamento, frequência e ticket médio por cliente
customer_metrics = orders.groupby('customer_id').agg(
    faturamento_total=('total', 'sum'),
    frequencia=('id', 'count')
).reset_index()
customer_metrics['ticket_medio'] = (
    customer_metrics['faturamento_total'] / customer_metrics['frequencia']
).round(2)

# Diversidade de categorias por cliente (mesma cadeia de chaves do SQL)
oi_full = (
    order_items
    .merge(orders[['id', 'customer_id']], left_on='order_id', right_on='id', suffixes=('', '_order'))
    .merge(product_variants[['id', 'product_id']], left_on='product_variant_id', right_on='id', suffixes=('', '_pv'))
    .merge(products[['id', 'category_id']], left_on='product_id', right_on='id', suffixes=('', '_prod'))
)
diversidade = oi_full.groupby('customer_id')['category_id'].nunique().reset_index(
    name='diversidade_categorias'
)

resultado = customer_metrics.merge(diversidade, on='customer_id')

# Top 10 clientes fiéis via pandas
top10_pandas = resultado[resultado['diversidade_categorias'] >= 13].sort_values(
    by=['ticket_medio', 'customer_id'], ascending=[False, True]
).head(10)

print("Top 10 clientes fiéis (validação via pandas):")
display(top10_pandas[['customer_id', 'faturamento_total', 'frequencia', 'ticket_medio', 'diversidade_categorias']])

# Comparação SQL vs pandas
comparacao = top10_sql.merge(
    top10_pandas[['customer_id', 'ticket_medio']],
    on='customer_id',
    suffixes=('_sql', '_pandas')
)
comparacao['diferenca'] = (comparacao['ticket_medio_sql'] - comparacao['ticket_medio_pandas']).abs()

print("\nMaior diferença entre SQL e pandas:", comparacao['diferenca'].max())
print("(diferenças de centavo, se houver, vêm de precisão de ponto flutuante")
print("entre float64/numpy e NUMERIC do Postgres; não indicam erro de lógica)")
display(comparacao)

# Categoria mais vendida entre os 10 clientes fiéis
top10_ids = top10_pandas['customer_id'].tolist()
oi_top10 = oi_full[oi_full['customer_id'].isin(top10_ids)]
oi_top10 = oi_top10.merge(categories[['id', 'name']], left_on='category_id', right_on='id', suffixes=('', '_cat'))

cat_ranking = oi_top10.groupby('name')['quantity'].sum().sort_values(ascending=False)
print("\nCategoria com maior soma de itens comprados entre os 10 clientes fiéis:")
print(cat_ranking.head(5))

Top 10 clientes fiéis (validação via pandas):


,customer_id,faturamento_total,frequencia,ticket_medio,diversidade_categorias
21,22,1087838.44,26,41839.94,14
1476,1477,916262.58,22,41648.30,14
928,929,1082775.89,26,41645.23,14
1115,1116,655737.20,16,40983.57,14
1690,1691,815471.30,20,40773.56,14
773,774,726127.99,18,40340.44,14
1469,1470,1040553.09,26,40021.27,14
1598,1599,997616.46,25,39904.66,14
964,965,677297.78,17,39841.05,14
1721,1722,1146455.22,29,39532.94,14



Maior diferença entre SQL e pandas: 0.010000000002037268
(diferenças de centavo, se houver, vêm de precisão de ponto flutuante
entre float64/numpy e NUMERIC do Postgres; não indicam erro de lógica)


,customer_id,faturamento_total,frequencia,ticket_medio_sql,diversidade_categorias,ticket_medio_pandas,diferenca
0,22,1087838.44,26,41839.94,14,41839.94,0.00
1,1477,916262.58,22,41648.30,14,41648.30,0.00
2,929,1082775.89,26,41645.23,14,41645.23,0.00
3,1116,655737.20,16,40983.58,14,40983.57,0.01
4,1691,815471.30,20,40773.57,14,40773.56,0.01
5,774,726127.99,18,40340.44,14,40340.44,0.00
6,1470,1040553.09,26,40021.27,14,40021.27,0.00
7,1599,997616.46,25,39904.66,14,39904.66,0.00
8,965,677297.78,17,39841.05,14,39841.05,0.00
9,1722,1146455.22,29,39532.94,14,39532.94,0.00



Categoria com maior soma de itens comprados entre os 10 clientes fiéis:
name
Hélices                492
Coletes Salva-Vidas    393
Eletrônica Náutica     392
Âncoras                387
Iluminação             333
Name: quantity, dtype: int64


## Questão 4.2 - Explicação

**1. Como você chegou nas categorias mais vendidas? (mapeamento da cadeia de chaves)**

Usei `orders.customer_id` para identificar o cliente e percorri as relações `orders.id = order_items.order_id`, `order_items.product_variant_id = product_variants.id`, `product_variants.product_id = products.id` e `products.category_id = categories.id`. A categoria não está diretamente ligada ao pedido, mas ao produto vendido em cada item.

**2. Qual lógica utilizou para filtrar os clientes com diversidade mínima?**

Agreguei `COUNT(DISTINCT category_id)` por cliente e apliquei `WHERE diversidade_categorias >= 13` sobre essa agregação, garantindo que só entrem no ranking clientes que de fato navegaram por 13 ou mais categorias diferentes.

**3. Como garantiu que a contagem de itens refletisse apenas o Top 10?**

Isolei os 10 clientes fiéis em uma CTE (`ORDER BY ticket_medio DESC, customer_id ASC LIMIT 10`) e só então fiz o `JOIN` com `order_items`. Assim, a soma de quantidade reflete exclusivamente as compras desses 10 clientes, não da base inteira.